# 01_Summarize

환자별 Peak detection 지표(Precision/Recall/F1), Apnea 추정 횟수, Apnea 평균 길이(초), 호흡간 길이(초)를 요약합니다.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DETECTED_DIR = Path('/home/jhkim/NAVA/03_LABEL_BREATH/stored_results/00_detected/20260226')
FS_DEFAULT = 100

if not DETECTED_DIR.exists():
    raise FileNotFoundError(f'Directory not found: {DETECTED_DIR}')

files = sorted(DETECTED_DIR.glob('movingwinddetected_patient_*.xlsx'))
if not files:
    raise FileNotFoundError(f'No detected files found in: {DETECTED_DIR}')

print(f'Found {len(files)} patient files in {DETECTED_DIR}')


In [ ]:
def normalize_patient_id(v: str) -> str:
    m = re.search(r'(\d+)', str(v))
    if not m:
        return str(v)
    return f"{int(m.group(1)):02d}"

def infer_time_seconds(ts: np.ndarray, fs: float = FS_DEFAULT) -> np.ndarray:
    ts = np.asarray(ts, dtype=float)
    if len(ts) < 3:
        return np.arange(len(ts), dtype=float) / fs

    d = np.diff(ts)
    d = d[np.isfinite(d) & (d > 0)]
    if len(d) == 0:
        return np.arange(len(ts), dtype=float) / fs

    dt = float(np.median(d))
    if 0.005 <= dt <= 0.02:
        return ts
    if 5.0 <= dt <= 20.0:
        return ts / 1000.0
    if 0.5 <= dt <= 2.0:
        return ts / fs
    return np.arange(len(ts), dtype=float) / fs

def contiguous_true_runs(mask: np.ndarray):
    mask = np.asarray(mask, dtype=bool)
    n = len(mask)
    i = 0
    runs = []
    while i < n:
        if not mask[i]:
            i += 1
            continue
        j = i
        while j < n and mask[j]:
            j += 1
        runs.append((i, j))
        i = j
    return runs

def parse_params_sheet(params_df: pd.DataFrame) -> dict:
    p = {}
    if {'parameter', 'value'}.issubset(params_df.columns):
        for k, v in zip(params_df['parameter'], params_df['value']):
            p[str(k)] = v
    return p


In [ ]:
rows = []

for f in files:
    data_df = pd.read_excel(f, sheet_name='data')
    params_df = pd.read_excel(f, sheet_name='params')
    params = parse_params_sheet(params_df)

    pid = normalize_patient_id(params.get('patient_id', f.stem))

    timestamp = pd.to_numeric(data_df['timestamp'], errors='coerce').to_numpy()
    t_sec = infer_time_seconds(timestamp, fs=float(params.get('fs_hz', FS_DEFAULT)))

    apnea_mask = data_df['apnea'].astype(bool).to_numpy() if 'apnea' in data_df.columns else np.zeros(len(data_df), dtype=bool)
    apnea_runs = contiguous_true_runs(apnea_mask)

    dt = np.diff(t_sec)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    sample_dt = float(np.median(dt)) if len(dt) else 1.0 / float(params.get('fs_hz', FS_DEFAULT))

    apnea_lengths = []
    for s, e in apnea_runs:
        dur = (t_sec[e - 1] - t_sec[s]) + sample_dt
        apnea_lengths.append(float(max(dur, 0.0)))

    detected_peak_mask = data_df['detected_peak'].astype(bool).to_numpy() if 'detected_peak' in data_df.columns else np.zeros(len(data_df), dtype=bool)
    detected_idx = np.flatnonzero(detected_peak_mask)
    if len(detected_idx) >= 2:
        breath_intervals = np.diff(t_sec[detected_idx])
        breath_intervals = breath_intervals[np.isfinite(breath_intervals) & (breath_intervals > 0)]
    else:
        breath_intervals = np.array([], dtype=float)

    rows.append({
        'patient_id': pid,
        'peak_precision': float(params.get('eval_precision', np.nan)),
        'peak_recall': float(params.get('eval_recall', np.nan)),
        'peak_f1': float(params.get('eval_f1_score', np.nan)),
        'apnea_count': int(len(apnea_runs)),
        'apnea_mean_length_sec': float(np.mean(apnea_lengths)) if apnea_lengths else np.nan,
        'breath_interval_mean_sec': float(np.mean(breath_intervals)) if len(breath_intervals) else np.nan,
    })

summary_df = pd.DataFrame(rows).sort_values('patient_id').reset_index(drop=True)
summary_df


In [ ]:
summary_view = summary_df.copy()
for c in ['peak_precision', 'peak_recall', 'peak_f1', 'apnea_mean_length_sec', 'breath_interval_mean_sec']:
    summary_view[c] = summary_view[c].round(4)
summary_view
